# Ancient AMP resurrection — Stage 1 to 6 (hepcidin)

**Project:** Deep-time resurrection of ancestral iron-withholding proteins to recover lost antimicrobial peptides against Gram-negative pathogens.

This notebook is a **scaffold**, not a finished analysis. It gets you from sequences to reconstructed ancestral hepcidins so you can see the method work end to end. The scientific quality lives in **Stage 2 (curation)**: a keyword grab is not a clean ortholog set, and a tree built on messy sequences gives meaningless ancestors. Curate with your mentor before trusting any resurrected peptide.

**Run this in Google Colab** (Runtime for the tools, browser for you). It does not touch your laptop.

Precedent to cite: Maasch, Torres, Melo & de la Fuente-Nunez (2023), *Cell Host & Microbe*, 31(8), 1260–1274. Your novelty: they mined peptides from extinct proteomes; you reconstruct the ancestral iron-withholding proteins themselves and derive peptides from those.

## Stage 0 — Install tools
MAFFT for alignment, IQ-TREE for the tree and ancestral reconstruction, Biopython for fetching.

In [ ]:
!apt-get -qq install -y mafft iqtree >/dev/null
!pip -q install biopython >/dev/null
import Bio, subprocess
print("Biopython", Bio.__version__)
print(subprocess.run(["iqtree","--version"],capture_output=True,text=True).stdout.splitlines()[0])
print("mafft", subprocess.run(["mafft","--version"],capture_output=True,text=True).stderr.strip())

## Stage 1 — Fetch sequences
Set your email (NCBI requires it). The query grabs hepcidin protein sequences across vertebrates. This is a **first pass**: it will include partials, isoforms and duplicates that Stage 2 must clean.

In [ ]:
from Bio import Entrez, SeqIO
from io import StringIO

Entrez.email = "YOUR_EMAIL@example.com"   # <-- put your email here
# Optional but faster/more reliable: Entrez.api_key = "your_ncbi_api_key"

QUERY = ('hepcidin[Protein Name] AND vertebrata[Organism] '
         'AND 60:120[Sequence Length] NOT partial[Properties]')
RETMAX = 600

h = Entrez.esearch(db="protein", term=QUERY, retmax=RETMAX)
ids = Entrez.read(h)["IdList"]; h.close()
print("hits:", len(ids))

fetch = Entrez.efetch(db="protein", id=ids, rettype="fasta", retmode="text")
records = list(SeqIO.parse(StringIO(fetch.read()), "fasta")); fetch.close()
print("downloaded records:", len(records))

## Stage 2 — Curate  ← the scientific step
Keep one sequence per species, drop very short/very long outliers, drop exact duplicates. **This is deliberately simple.** For a real analysis you and your mentor should: confirm each is orthologous hepcidin (HAMP), not a paralog; check the taxon sampling spans jawed vertebrates evenly (fish, amphibians, reptiles, birds, mammals) so the deep root is supported; and remove misannotated entries by eye.

In [ ]:
import re
def organism(rec):
    m = re.search(r"\[(.*?)\]", rec.description)
    return m.group(1) if m else rec.id

seen, curated = set(), []
for rec in sorted(records, key=lambda r: len(r.seq), reverse=True):
    org = organism(rec)
    seq = str(rec.seq).replace("X","").replace("*","")
    if org in seen: continue
    if not (60 <= len(seq) <= 120): continue
    seen.add(org)
    rec.id = re.sub(r"[^A-Za-z0-9]","_", org)[:40]
    rec.description = ""
    curated.append(rec)

print("species kept:", len(curated))
SeqIO.write(curated, "hepcidin.fasta", "fasta")
for r in curated[:10]: print(r.id, len(r.seq))

## Stage 3 — Align (MAFFT)
L-INS-i is accurate for a few hundred short sequences.

In [ ]:
!mafft --localpair --maxiterate 1000 --quiet hepcidin.fasta > hepcidin_aln.fasta
from Bio import AlignIO
aln = AlignIO.read("hepcidin_aln.fasta","fasta")
print("alignment:", len(aln), "sequences x", aln.get_alignment_length(), "columns")

## Stage 4 & 5 — Tree + ancestral reconstruction (IQ-TREE)
`-asr` computes marginal ancestral sequences at every internal node. `-m TEST` picks the best substitution model. `-bb 1000` gives branch support so you know which ancestors to trust.

In [ ]:
!iqtree -s hepcidin_aln.fasta -m TEST -asr -bb 1000 -nt AUTO -pre hepcidin -redo
print("\nfiles produced:")
!ls -1 hepcidin.*

## Stage 6 — Read the ancestral sequences
IQ-TREE writes per-node ancestral states to `hepcidin.state`. The tree file names the nodes. Below we pull each internal node's most-likely sequence and its mean posterior probability, so you can rank ancestors by confidence. The **deepest well-supported node** is your candidate ancient hepcidin; its mature-peptide region is what you take forward to activity and structure prediction (Stage 7, a separate GPU notebook).

In [ ]:
import pandas as pd
df = pd.read_csv("hepcidin.state", sep="\t", comment="#")
# columns: Node, Site, State, p_A, p_R, ...  -> most-likely state per site
prob_cols = [c for c in df.columns if c.startswith("p_")]
df["maxp"] = df[prob_cols].max(axis=1)

anc = {}
for node, g in df.groupby("Node"):
    g = g.sort_values("Site")
    seq = "".join(g["State"].tolist())
    anc[node] = (seq.replace("-",""), g["maxp"].mean())

ranked = sorted(anc.items(), key=lambda kv: kv[1][1], reverse=True)
print("internal nodes reconstructed:", len(anc))
print("\ntop nodes by mean posterior probability:")
for node,(seq,conf) in ranked[:5]:
    print(f"{node}  conf={conf:.3f}  len={len(seq)}\n  {seq}\n")

with open("ancestral_hepcidins.fasta","w") as f:
    for node,(seq,conf) in ranked:
        f.write(f">{node}_conf{conf:.3f}\n{seq}\n")
print("written: ancestral_hepcidins.fasta")

## What you have, and what comes next

**You now have** reconstructed ancestral hepcidins with confidence scores, and a tree telling you how deep each node sits. Open `hepcidin.treefile` in [iTOL](https://itol.embl.de) to see which nodes are the deep jawed-vertebrate ancestors that justify the deep-time claim.

**Before trusting anything:**
- Redo Stage 2 properly with your mentor. Bad orthology = meaningless ancestors.
- A node is only worth resurrecting if its branch support (Stage 4) and its mean posterior probability (Stage 6) are both high.
- Reconstruct with more than one substitution model and check the deep ancestors agree; disagreement means low confidence, and you should say so honestly.

**Stage 7 (separate notebook, needs a GPU runtime):**
- Predict antimicrobial activity of each ancestral mature peptide with an AMP classifier.
- Fold the ancestral proteins with ESMFold or ColabFold to check the structure is plausible.
- Compare against living hepcidins to identify peptides truly lost in extant species.

**Then repeat the whole pipeline for the transferrin superfamily**, which reaches even deeper and carries lactoferricin as one mammalian branch.

Tell me when Stage 1–6 runs cleanly and I'll build the Stage 7 activity-and-structure notebook.